In [9]:
import pandas as pd
import numpy as np

v4 = pd.read_csv("/Users/othree/AIsummit/data/v4.csv")
print(f"Shape: {v4.shape}")
print(f"Columns: {v4.shape[1]} (facilities: 51, join keys: 3, nfhs: 108)")

Shape: (10088, 162)
Columns: 162 (facilities: 51, join keys: 3, nfhs: 108)


/var/folders/zb/yqyg0k1554l0vj7zf6mzbm240000gn/T/ipykernel_29067/220187587.py:4: DtypeWarning: Columns (28,29,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  v4 = pd.read_csv("/Users/othree/AIsummit/data/v4.csv")


In [10]:
## 1. 조인 커버리지

total = len(v4)
has_pincode = v4['pincode_clean'].notna().sum()
has_district = v4['district_clean'].notna().sum()
has_nfhs = v4['households_surveyed'].notna().sum()  # nfhs 조인 성공 여부 proxy

print(f"전체 시설: {total:,}")
print(f"pincode 있음 (facilities → pincode 매칭): {has_pincode:,} ({has_pincode/total*100:.1f}%)")
print(f"district 있음 (pincode → nfhs 키 확보):   {has_district:,} ({has_district/total*100:.1f}%)")
print(f"nfhs 데이터 붙음 (최종 3-way 조인):        {has_nfhs:,} ({has_nfhs/total*100:.1f}%)")
print()
print(f"nfhs 미매칭 (pincode 없거나 district 불일치): {total - has_nfhs:,}")

전체 시설: 10,088
pincode 있음 (facilities → pincode 매칭): 9,926 (98.4%)
district 있음 (pincode → nfhs 키 확보):   9,700 (96.2%)
nfhs 데이터 붙음 (최종 3-way 조인):        9,499 (94.2%)

nfhs 미매칭 (pincode 없거나 district 불일치): 589


In [11]:
## 2. 시설 분포 — state별

state_counts = v4['state_clean'].value_counts()
print(f"State 수: {state_counts.nunique()}")
print()
print(state_counts.to_string())

State 수: 31

state_clean
maharashtra                                 1592
gujarat                                      956
uttar pradesh                                943
tamil nadu                                   811
karnataka                                    517
kerala                                       511
west bengal                                  478
punjab                                       475
haryana                                      460
telangana                                    458
rajasthan                                    394
delhi                                        339
madhya pradesh                               308
andhra pradesh                               307
bihar                                        260
jharkhand                                    154
chhattisgarh                                 136
uttarakhand                                  128
odisha                                       118
assam                                       

In [12]:
## 3. 시설 분포 — organization_type / facilityTypeId / operatorTypeId

for col in ['organization_type', 'facilityTypeId', 'operatorTypeId']:
    print(f"[{col}]")
    print(v4[col].value_counts().head(15).to_string())
    print()

[organization_type]
organization_type
facility                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [13]:
## 4. 시설 수치형 컬럼 분포

num_cols = ['numberDoctors', 'capacity', 'yearEstablished',
            'distinct_social_media_presence_count', 'number_of_facts_about_the_organization',
            'engagement_metrics_n_followers']

print(v4[num_cols].describe().T.to_string())

                                         count      mean        std  min  25%  50%  75%         max
distinct_social_media_presence_count    9966.0  3.456204   2.087356  0.0  2.0  4.0  5.0   79.525787
number_of_facts_about_the_organization  2757.0  6.727965  13.870827  0.0  3.0  5.0  7.0  442.000000


In [14]:
## 5. NFHS 핵심 지표 분포 (district별 통계)

nfhs_key = [
    'institutional_birth_5y_pct',
    'births_attended_by_skilled_hp_5y_10_pct',
    'child_12_23m_fully_vaccinated_based_on_information_from_eit_pct',
    'child_u5_who_are_stunted_height_for_age_18_pct',
    'child_u5_who_are_underweight_weight_for_age_18_pct',
    'all_w15_49_who_are_anaemic_pct',
    'hh_improved_water_pct',
    'hh_use_improved_sanitation_pct',
    'fp_cm_w15_49_modern_method_pct',
    'w20_24_married_before_age_18_years_pct',
]

nfhs_district = (
    v4[['state_clean', 'district_clean'] + nfhs_key]
    .drop_duplicates(subset=['state_clean', 'district_clean'])
    .dropna(subset=['district_clean'])
)
print(f"District 수 (nfhs 있는 것): {len(nfhs_district)}")
print()
print(nfhs_district[nfhs_key].describe().T[['mean','std','min','50%','max']].round(1).to_string())

District 수 (nfhs 있는 것): 555

                                         mean   std   min   50%    max
institutional_birth_5y_pct               90.7   9.4  34.8  93.2  100.0
births_attended_by_skilled_hp_5y_10_pct  91.2   8.1  39.2  93.0  100.0
all_w15_49_who_are_anaemic_pct           56.0  10.6  25.5  56.8   84.8
hh_improved_water_pct                    95.3   6.6  41.2  97.8  100.0
hh_use_improved_sanitation_pct           72.5  13.8  29.2  74.2   99.9
fp_cm_w15_49_modern_method_pct           56.0  13.1  10.6  56.6   81.2


In [15]:
## 6. district별 시설 수 vs NFHS 핵심 지표 상관관계

fac_per_district = v4.groupby('district_clean').size().reset_index(name='facility_count')
nfhs_corr = nfhs_district.merge(fac_per_district, on='district_clean', how='left')

print("시설 수 vs 핵심 지표 상관관계 (Pearson r):")
print()
corr = nfhs_corr[['facility_count'] + nfhs_key].corr()['facility_count'].drop('facility_count').sort_values(ascending=False)
for col, r in corr.items():
    bar = '█' * int(abs(r) * 20)
    sign = '+' if r > 0 else '-'
    print(f"  {sign}{bar:<20} {r:+.3f}  {col}")

시설 수 vs 핵심 지표 상관관계 (Pearson r):



ValueError: could not convert string to float: '(82.1)'

In [16]:
## 7. 결측치 요약 (주요 컬럼)

key_cols = ['pincode_clean', 'state_clean', 'district_clean',
            'numberDoctors', 'capacity', 'yearEstablished',
            'latitude', 'longitude'] + nfhs_key

missing = v4[key_cols].isnull().sum()
pct = (missing / len(v4) * 100).round(1)
print(pd.DataFrame({'missing': missing, 'missing_%': pct}).to_string())

                                                                 missing  missing_%
pincode_clean                                                        162        1.6
state_clean                                                          388        3.8
district_clean                                                       388        3.8
numberDoctors                                                       6455       64.0
capacity                                                            7568       75.0
yearEstablished                                                     5284       52.4
latitude                                                             118        1.2
longitude                                                            118        1.2
institutional_birth_5y_pct                                           589        5.8
births_attended_by_skilled_hp_5y_10_pct                              589        5.8
child_12_23m_fully_vaccinated_based_on_information_from_eit_pct      589    

In [17]:
## 8. State별 핵심 지표 평균 (상위/하위 5개 state)

state_avg = (
    v4[['state_clean', 'district_clean'] + nfhs_key]
    .drop_duplicates(subset=['state_clean', 'district_clean'])
    .groupby('state_clean')[nfhs_key]
    .mean()
    .round(1)
)

target = 'institutional_birth_5y_pct'
print(f"[병원 출산율] 상위 5 state:")
print(state_avg[target].nlargest(5).to_string())
print()
print(f"[병원 출산율] 하위 5 state:")
print(state_avg[target].nsmallest(5).to_string())
print()
target2 = 'child_u5_who_are_stunted_height_for_age_18_pct'
print(f"[발육부진율] 상위 5 state (심각):")
print(state_avg[target2].nlargest(5).to_string())

TypeError: agg function failed [how->mean,dtype->object]